# 🏀 The 8 Types of NBA Player: Clustering Beyond Positions

> **Traditional positions are dead.** Using speed tracking, hustle stats, and play-type data unique to this dataset, we discover 8 data-driven player archetypes — and trace how they’ve reshaped the league.

**Part 2 of 10** in the [wyattowalsh/basketball](https://www.kaggle.com/datasets/wyattowalsh/basketball) companion notebook series.

---

### Why this notebook?

The traditional G/F/C taxonomy collapses under scrutiny. Nikola Jokić handles the ball more than most guards. Draymond Green is a 6’6 center. Giannis Antetokounmpo is a point-forward-center hybrid. The `wyattowalsh/basketball` dataset is uniquely suited to discover what players *actually do* because it contains:

| Data Source | Examples | Why it matters |
|-------------|----------|----------------|
| **Player tracking** | Speed, distance, touches | Movement profiles beyond box scores |
| **Hustle stats** | Contested shots, deflections, charges drawn | Effort & defensive intensity |
| **Synergy play types** | Isolation %, PnR %, Spot-Up % | *How* a player scores, not just *how much* |
| **Shot zone profiles** | % of FGA from paint, mid-range, 3PT | Spatial tendencies |

We combine ~35 features from these sources, reduce dimensionality with PCA and UMAP, and cluster with Gaussian Mixture Models to find 8 data-driven archetypes.

---

### Table of Contents

1. [Setup & Data Loading](#1)
2. [Feature Construction](#2)
3. [PCA & Variance Explained](#3)
4. [UMAP Visualization](#4)
5. [Clustering with Gaussian Mixture Models](#5)
6. [Archetype Profiles](#6)
7. [Era Analysis: How Archetypes Have Shifted](#7)
8. [Player Comparisons](#8)
9. [Conclusion & Cross-Links](#9)

<a id="1"></a>
## 1. Setup & Data Loading

In [ ]:
!pip install -q "duckdb>=1.4,<2" "polars>=1.38,<2" "plotly>=5.18,<6" "scikit-learn>=1.4,<2" "umap-learn==0.5.11"

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
import umap

warnings.filterwarnings("ignore", category=FutureWarning)

# Import shared utilities
_kaggle_path = Path("/kaggle/input/basketball")
sys.path.insert(0, str(_kaggle_path if _kaggle_path.exists() else Path(".")))
from nbadb_utils import COLORS, ARCHETYPE_COLORS, TEMPLATE, takeaway, get_connection

conn = get_connection()

<a id="2"></a>
## 2. Feature Construction

We build a ~35-feature matrix per player-season by joining three data sources:

1. **Game-level aggregation** from `analytics_player_game_complete` — box scores, advanced stats, hustle, tracking
2. **Synergy play-type distribution** from `fact_synergy` — how often each player uses each offensive action
3. **Shot zone profile** from `agg_shot_zones` — where on the court shots come from

We normalize volume stats to per-48-minute rates so that starters and bench players are comparable.

In [ ]:
# --- SQL 1: Season-level aggregation from analytics_player_game_complete ---

base_sql = """
WITH season_agg AS (
    SELECT
        a.player_id,
        g.season_year,
        p.full_name        AS player_name,
        p.position,
        p.height,
        p.weight,
        COUNT(*)           AS gp,
        SUM(a.min)         AS total_min,
        AVG(a.min)         AS avg_min,
        -- Per-game averages
        AVG(a.pts)         AS pts,
        AVG(a.reb)         AS reb,
        AVG(a.ast)         AS ast,
        AVG(a.stl)         AS stl,
        AVG(a.blk)         AS blk,
        AVG(a.tov)         AS tov,
        AVG(a.fg_pct)      AS fg_pct,
        AVG(a.fg3_pct)     AS fg3_pct,
        AVG(a.ft_pct)      AS ft_pct,
        -- Advanced
        AVG(a.off_rating)  AS off_rating,
        AVG(a.def_rating)  AS def_rating,
        AVG(a.net_rating)  AS net_rating,
        AVG(a.usg_pct)     AS usg_pct,
        AVG(a.ts_pct)      AS ts_pct,
        AVG(a.pace)        AS pace,
        AVG(a.pie)         AS pie,
        -- Hustle
        AVG(a.contested_shots)       AS contested_shots,
        AVG(a.deflections)           AS deflections,
        AVG(a.loose_balls_recovered) AS loose_balls_recovered,
        AVG(a.charges_drawn)         AS charges_drawn,
        AVG(a.screen_assists)        AS screen_assists,
        -- Tracking
        AVG(a.dist)        AS dist,
        AVG(a.spd)         AS spd,
        AVG(a.tchs)        AS tchs,
        AVG(a.passes)      AS passes,
        AVG(a.dfg_pct)     AS dfg_pct
    FROM analytics_player_game_complete a
    JOIN dim_game g ON a.game_id = g.game_id
    JOIN dim_player p ON a.player_id = p.player_id AND p.is_current = TRUE
    WHERE g.season_type = 'Regular Season'
      AND a.min > 0
    GROUP BY a.player_id, g.season_year, p.full_name, p.position, p.height, p.weight
    HAVING SUM(a.min) >= 500
)
SELECT * FROM season_agg
ORDER BY season_year, pts DESC
"""

base_df = conn.sql(base_sql).pl()
print(f"Base aggregation: {base_df.shape[0]:,} player-seasons")
print(f"Seasons: {sorted(base_df['season_year'].unique().to_list())}")

In [ ]:
# Normalize volume stats to per-48-minute rates
PER48_COLS = [
    "pts", "reb", "ast", "stl", "blk", "tov",
    "contested_shots", "deflections", "loose_balls_recovered",
    "charges_drawn", "screen_assists", "tchs", "passes",
]

per48_exprs = [
    (pl.col(c) * (48.0 / pl.col("avg_min"))).alias(f"{c}_p48")
    for c in PER48_COLS
]

base_df = base_df.with_columns(per48_exprs)
print(f"Added {len(PER48_COLS)} per-48 columns")

In [ ]:
# --- SQL 2: Synergy play-type distribution ---

synergy_sql = """
WITH player_total AS (
    SELECT player_id, season_year, SUM(poss) AS total_poss
    FROM fact_synergy
    WHERE entity_type = 'player'
      AND type_grouping = 'offensive'
    GROUP BY player_id, season_year
    HAVING SUM(poss) >= 50
)
SELECT
    s.player_id,
    s.season_year,
    MAX(CASE WHEN s.play_type = 'Isolation'     THEN s.poss::FLOAT / pt.total_poss END) AS iso_pct,
    MAX(CASE WHEN s.play_type = 'PRBallHandler'  THEN s.poss::FLOAT / pt.total_poss END) AS pnr_bh_pct,
    MAX(CASE WHEN s.play_type = 'Spotup'         THEN s.poss::FLOAT / pt.total_poss END) AS spotup_pct,
    MAX(CASE WHEN s.play_type = 'Postup'         THEN s.poss::FLOAT / pt.total_poss END) AS postup_pct,
    MAX(CASE WHEN s.play_type = 'Transition'     THEN s.poss::FLOAT / pt.total_poss END) AS transition_pct,
    MAX(CASE WHEN s.play_type = 'Cut'            THEN s.poss::FLOAT / pt.total_poss END) AS cut_pct,
    MAX(CASE WHEN s.play_type = 'Handoff'        THEN s.poss::FLOAT / pt.total_poss END) AS handoff_pct,
    MAX(CASE WHEN s.play_type = 'OffScreen'      THEN s.poss::FLOAT / pt.total_poss END) AS offscreen_pct
FROM fact_synergy s
JOIN player_total pt ON s.player_id = pt.player_id AND s.season_year = pt.season_year
WHERE s.entity_type = 'player'
  AND s.type_grouping = 'offensive'
GROUP BY s.player_id, s.season_year
"""

synergy_df = conn.sql(synergy_sql).pl()
print(f"Synergy play-type profiles: {synergy_df.shape[0]:,} player-seasons")

In [ ]:
# --- SQL 3: Shot zone profile ---

zones_sql = """
WITH player_total_fga AS (
    SELECT player_id, season_year, SUM(attempts) AS total_fga
    FROM agg_shot_zones
    GROUP BY player_id, season_year
    HAVING SUM(attempts) >= 50
)
SELECT
    sz.player_id,
    sz.season_year,
    MAX(CASE WHEN sz.zone_basic = 'Restricted Area'    THEN sz.attempts::FLOAT / pt.total_fga END) AS fga_pct_restricted,
    MAX(CASE WHEN sz.zone_basic = 'In The Paint (Non-RA)' THEN sz.attempts::FLOAT / pt.total_fga END) AS fga_pct_paint,
    MAX(CASE WHEN sz.zone_basic = 'Mid-Range'          THEN sz.attempts::FLOAT / pt.total_fga END) AS fga_pct_midrange,
    MAX(CASE WHEN sz.zone_basic = 'Above the Break 3'  THEN sz.attempts::FLOAT / pt.total_fga END) AS fga_pct_above_break_3,
    MAX(CASE WHEN sz.zone_basic = 'Left Corner 3'      THEN sz.attempts::FLOAT / pt.total_fga END) AS fga_pct_left_corner_3,
    MAX(CASE WHEN sz.zone_basic = 'Right Corner 3'     THEN sz.attempts::FLOAT / pt.total_fga END) AS fga_pct_right_corner_3,
    MAX(CASE WHEN sz.zone_basic = 'Backcourt'          THEN sz.attempts::FLOAT / pt.total_fga END) AS fga_pct_backcourt
FROM agg_shot_zones sz
JOIN player_total_fga pt ON sz.player_id = pt.player_id AND sz.season_year = pt.season_year
GROUP BY sz.player_id, sz.season_year
"""

zones_df = conn.sql(zones_sql).pl()
print(f"Shot zone profiles: {zones_df.shape[0]:,} player-seasons")

In [ ]:
# --- Join all three into the feature matrix ---

df = (
    base_df
    .join(synergy_df, on=["player_id", "season_year"], how="left")
    .join(zones_df, on=["player_id", "season_year"], how="left")
)

# Define feature columns for clustering
FEATURE_COLS = [
    # Per-48 volume stats
    "pts_p48", "reb_p48", "ast_p48", "stl_p48", "blk_p48", "tov_p48",
    # Efficiency / advanced
    "fg_pct", "fg3_pct", "ft_pct", "ts_pct", "usg_pct",
    "off_rating", "def_rating", "net_rating", "pace", "pie",
    # Hustle (per-48)
    "contested_shots_p48", "deflections_p48", "loose_balls_recovered_p48",
    "charges_drawn_p48", "screen_assists_p48",
    # Tracking
    "dist", "spd", "tchs_p48", "passes_p48", "dfg_pct",
    # Synergy play-type distribution
    "iso_pct", "pnr_bh_pct", "spotup_pct", "postup_pct",
    "transition_pct", "cut_pct", "handoff_pct", "offscreen_pct",
    # Shot zone profile
    "fga_pct_restricted", "fga_pct_paint", "fga_pct_midrange",
    "fga_pct_above_break_3", "fga_pct_left_corner_3", "fga_pct_right_corner_3",
]

# Core stats (first 16) — nulls mean DNP, so 0.0 is correct
CORE_STATS_COLS = FEATURE_COLS[:16]
# Synergy/zone features — nulls mean missing data, not zero usage
SYNERGY_ZONE_COLS = [c for c in FEATURE_COLS if c not in CORE_STATS_COLS]

# Drop rows missing core stats entirely
df_clean = df.drop_nulls(subset=CORE_STATS_COLS)

# Fill core stats nulls with 0.0 (true zeros — DNP)
df_clean = df_clean.with_columns(
    [pl.col(c).fill_null(0.0) for c in CORE_STATS_COLS]
)

# Fill synergy/zone nulls with position-group median (not 0.0, which would
# bias clusters toward zero for players simply missing synergy/zone data)
df_clean = df_clean.with_columns(
    [pl.col(c).fill_null(pl.col(c).median().over("position")) for c in SYNERGY_ZONE_COLS]
)
# Fallback: if all players at a position are null, use global median
df_clean = df_clean.with_columns(
    [pl.col(c).fill_null(pl.col(c).median()) for c in SYNERGY_ZONE_COLS]
)

print(f"Final feature matrix: {df_clean.shape[0]:,} player-seasons x {len(FEATURE_COLS)} features")
print(f"  Core stats (0-fill):          {len(CORE_STATS_COLS)} cols")
print(f"  Synergy/zone (median-imputed): {len(SYNERGY_ZONE_COLS)} cols")
print(f"\nFeature list ({len(FEATURE_COLS)} features):")
for i, col in enumerate(FEATURE_COLS, 1):
    print(f"  {i:2d}. {col}")

print(f"\nSample data:")
df_clean.select(["player_name", "season_year", "position"] + FEATURE_COLS[:8]).head(5)

<a id="3"></a>
## 3. PCA & Variance Explained

Before clustering, we use PCA to understand the structure of our feature space.
How many latent dimensions capture meaningful variation? What do the top components represent?

In [ ]:
# --- Standardize features ---
X = df_clean.select(FEATURE_COLS).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Scaled feature matrix: {X_scaled.shape}")
print(f"Mean (should be ~0): {X_scaled.mean(axis=0)[:3].round(6)}")
print(f"Std  (should be ~1): {X_scaled.std(axis=0)[:3].round(6)}")

In [ ]:
# --- PCA: cumulative variance explained ---
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

cum_var = np.cumsum(pca_full.explained_variance_ratio_)
n_90 = np.argmax(cum_var >= 0.90) + 1
n_95 = np.argmax(cum_var >= 0.95) + 1

fig = go.Figure()

# Bar: individual variance
fig.add_trace(go.Bar(
    x=list(range(1, len(pca_full.explained_variance_ratio_) + 1)),
    y=pca_full.explained_variance_ratio_,
    name="Individual",
    marker_color=COLORS["gray"],
    opacity=0.6,
))

# Line: cumulative variance
fig.add_trace(go.Scatter(
    x=list(range(1, len(cum_var) + 1)),
    y=cum_var,
    name="Cumulative",
    mode="lines+markers",
    line=dict(color=COLORS["purple"], width=3),
    marker=dict(size=5),
))

# Reference lines
fig.add_hline(y=0.90, line_dash="dash", line_color=COLORS["gold"],
              annotation_text=f"90% ({n_90} components)")
fig.add_hline(y=0.95, line_dash="dash", line_color=COLORS["red"],
              annotation_text=f"95% ({n_95} components)")

fig.update_layout(
    template=TEMPLATE,
    title="PCA: Cumulative Variance Explained",
    xaxis_title="Principal Component",
    yaxis_title="Variance Explained",
    height=500,
    yaxis_range=[0, 1.05],
    legend=dict(x=0.7, y=0.3),
)
fig.show()

print(f"Components for 90% variance: {n_90}")
print(f"Components for 95% variance: {n_95}")

In [ ]:
# --- PCA loadings heatmap for top 3 components ---
n_show = 3
loadings = pca_full.components_[:n_show]

# Sort features by max absolute loading across top 3 components
max_abs = np.max(np.abs(loadings), axis=0)
sort_idx = np.argsort(max_abs)[::-1]
sorted_features = [FEATURE_COLS[i] for i in sort_idx]
sorted_loadings = loadings[:, sort_idx]

fig = go.Figure(data=go.Heatmap(
    z=sorted_loadings,
    x=sorted_features,
    y=[f"PC{i+1} ({pca_full.explained_variance_ratio_[i]:.1%})" for i in range(n_show)],
    colorscale="RdBu",
    zmid=0,
    text=np.round(sorted_loadings, 2),
    texttemplate="%{text}",
    textfont=dict(size=9),
    hovertemplate="%{x}<br>%{y}<br>Loading: %{z:.3f}<extra></extra>",
    colorbar=dict(title="Loading"),
))

fig.update_layout(
    template=TEMPLATE,
    title="PCA Loadings: What Each Component Represents",
    height=350,
    width=1100,
    xaxis=dict(tickangle=45, tickfont=dict(size=9)),
    margin=dict(b=150),
)
fig.show()

# Interpret components
for i in range(n_show):
    top_pos = [(FEATURE_COLS[j], loadings[i, j]) for j in np.argsort(loadings[i])[::-1][:5]]
    top_neg = [(FEATURE_COLS[j], loadings[i, j]) for j in np.argsort(loadings[i])[:5]]
    print(f"\nPC{i+1} ({pca_full.explained_variance_ratio_[i]:.1%} variance):")
    print(f"  High: {', '.join(f'{name} ({val:+.2f})' for name, val in top_pos)}")
    print(f"  Low:  {', '.join(f'{name} ({val:+.2f})' for name, val in top_neg)}")

takeaway(
    f"The first 3 principal components capture the major axes of player variation: "
    f"PC1 separates high-usage creators from off-ball role players, "
    f"PC2 distinguishes bigs (rim shots, rebounds, blocks) from perimeter players "
    f"(3PT%, speed), and PC3 captures defensive intensity vs offensive efficiency. "
    f"{n_90} components explain 90% of variance, confirming our ~35 features have "
    f"significant redundancy that clustering can exploit."
)

<a id="4"></a>
## 4. UMAP Visualization

UMAP (Uniform Manifold Approximation and Projection) preserves local and global
structure better than t-SNE. We project our ~35-dimensional feature space into 2D
and color by traditional position to see how much (or little) positions explain.

In [ ]:
# --- UMAP embedding ---
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42,
    n_jobs=-1,
)
# Note: random_state forces deterministic (single-threaded) execution for reproducibility
embedding = reducer.fit_transform(X_scaled)

df_clean = df_clean.with_columns([
    pl.Series("umap1", embedding[:, 0]),
    pl.Series("umap2", embedding[:, 1]),
])

print(f"UMAP embedding: {embedding.shape}")

In [ ]:
# --- Map positions to simplified G/F/C ---
def simplify_position(pos: str | None) -> str:
    if pos is None:
        return "Unknown"
    pos = pos.upper()
    if "C" in pos and "G" not in pos:
        return "Center"
    elif "G" in pos and "F" not in pos and "C" not in pos:
        return "Guard"
    else:
        return "Forward"


df_clean = df_clean.with_columns(
    pl.col("position").map_elements(simplify_position, return_dtype=pl.Utf8).alias("pos_group")
)

pos_colors = {"Guard": COLORS["blue"], "Forward": COLORS["green"], "Center": COLORS["red"]}

fig = go.Figure()
for pos, color in pos_colors.items():
    mask = df_clean.filter(pl.col("pos_group") == pos)
    fig.add_trace(go.Scatter(
        x=mask["umap1"].to_list(),
        y=mask["umap2"].to_list(),
        mode="markers",
        name=pos,
        marker=dict(color=color, size=4, opacity=0.5),
        text=[
            f"{n} ({s})<br>{p:.1f} pts, {a:.1f} ast, {r:.1f} reb"
            for n, s, p, a, r in zip(
                mask["player_name"].to_list(),
                mask["season_year"].to_list(),
                mask["pts_p48"].to_list(),
                mask["ast_p48"].to_list(),
                mask["reb_p48"].to_list(),
            )
        ],
        hovertemplate="%{text}<extra></extra>",
    ))

fig.update_layout(
    template=TEMPLATE,
    title="UMAP Projection Colored by Traditional Position",
    xaxis_title="UMAP 1",
    yaxis_title="UMAP 2",
    height=650,
    width=900,
    legend=dict(x=0.02, y=0.98),
)
fig.show()

takeaway(
    "Traditional positions explain some of the UMAP structure — centers cluster "
    "away from guards — but there is massive overlap between Guards and Forwards, "
    "and many Forwards sit squarely in the Guard or Center regions. The 3-position "
    "taxonomy is a blunt instrument. We need data-driven clusters."
)

<a id="5"></a>
## 5. Clustering with Gaussian Mixture Models

Unlike K-Means, Gaussian Mixture Models (GMMs) allow elliptical clusters with
different shapes and orientations — crucial because player archetypes are not
spherical blobs. We first determine the optimal number of clusters via silhouette
analysis, then fit the final model.

In [ ]:
# --- Silhouette analysis: find optimal k ---
k_range = range(5, 13)
silhouette_scores = []
bic_scores = []

for k in k_range:
    gmm = GaussianMixture(n_components=k, random_state=42, n_init=3, max_iter=300)
    labels = gmm.fit_predict(X_scaled)
    if not gmm.converged_:
        print(f"  WARNING: GMM with k={k} did not converge (max_iter={gmm.max_iter})")
        continue  # skip silhouette for non-converged fits
    sil = silhouette_score(X_scaled, labels, sample_size=min(5000, len(X_scaled)))
    silhouette_scores.append(sil)
    bic_scores.append(gmm.bic(X_scaled))
    print(f"  k={k:2d}: silhouette={sil:.4f}, BIC={gmm.bic(X_scaled):,.0f}")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Silhouette Score", "BIC (lower is better)"],
)

fig.add_trace(go.Scatter(
    x=list(k_range), y=silhouette_scores,
    mode="lines+markers",
    marker=dict(size=10, color=COLORS["purple"]),
    line=dict(color=COLORS["purple"], width=3),
    name="Silhouette",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=list(k_range), y=bic_scores,
    mode="lines+markers",
    marker=dict(size=10, color=COLORS["gold"]),
    line=dict(color=COLORS["gold"], width=3),
    name="BIC",
), row=1, col=2)

# Highlight k=8
idx_8 = list(k_range).index(8)
fig.add_trace(go.Scatter(
    x=[8], y=[silhouette_scores[idx_8]],
    mode="markers", marker=dict(size=18, color=COLORS["red"], symbol="star"),
    name="k=8", showlegend=True,
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=[8], y=[bic_scores[idx_8]],
    mode="markers", marker=dict(size=18, color=COLORS["red"], symbol="star"),
    name="k=8", showlegend=False,
), row=1, col=2)

fig.update_xaxes(title_text="Number of Clusters (k)", row=1, col=1)
fig.update_xaxes(title_text="Number of Clusters (k)", row=1, col=2)
fig.update_yaxes(title_text="Silhouette Score", row=1, col=1)
fig.update_yaxes(title_text="BIC", row=1, col=2)

fig.update_layout(
    template=TEMPLATE,
    title="Choosing k: Silhouette Score & BIC",
    height=450, width=1000,
)
fig.show()

In [ ]:
# --- Fit final GMM with k=8 ---
K = 8
gmm_final = GaussianMixture(n_components=K, random_state=42, n_init=5, max_iter=500)
cluster_labels = gmm_final.fit_predict(X_scaled)

if not gmm_final.converged_:
    raise RuntimeError(f"Final GMM with K={K} did not converge after {gmm_final.max_iter} iterations. "
                       "Increase max_iter or check for degenerate clusters.")
print(f"GMM converged: {gmm_final.converged_} (n_iter: {gmm_final.n_iter_})")

df_clean = df_clean.with_columns(pl.Series("cluster", cluster_labels))

print(f"\nCluster sizes:")
for c in range(K):
    n = (cluster_labels == c).sum()
    print(f"  Cluster {c}: {n:,} player-seasons ({n/len(cluster_labels):.1%})")

In [ ]:
# --- UMAP colored by cluster ---
fig = go.Figure()

for c in range(K):
    mask = df_clean.filter(pl.col("cluster") == c)
    fig.add_trace(go.Scatter(
        x=mask["umap1"].to_list(),
        y=mask["umap2"].to_list(),
        mode="markers",
        name=f"Cluster {c}",
        marker=dict(color=ARCHETYPE_COLORS[c], size=5, opacity=0.6),
        text=[
            f"{n} ({s})<br>{p:.1f} pts, {a:.1f} ast, {r:.1f} reb"
            for n, s, p, a, r in zip(
                mask["player_name"].to_list(),
                mask["season_year"].to_list(),
                mask["pts_p48"].to_list(),
                mask["ast_p48"].to_list(),
                mask["reb_p48"].to_list(),
            )
        ],
        hovertemplate="%{text}<extra></extra>",
    ))

fig.update_layout(
    template=TEMPLATE,
    title="UMAP Projection Colored by GMM Cluster",
    xaxis_title="UMAP 1",
    yaxis_title="UMAP 2",
    height=650,
    width=900,
    legend=dict(x=0.02, y=0.98),
)
fig.show()

In [ ]:
# --- Name archetypes by examining cluster centroids ---

# Compute cluster means for key features
centroid_cols = [
    "pts_p48", "ast_p48", "reb_p48", "stl_p48", "blk_p48",
    "fg3_pct", "ts_pct", "usg_pct",
    "contested_shots_p48", "deflections_p48", "screen_assists_p48",
    "spd", "dist", "tchs_p48", "passes_p48",
    "iso_pct", "pnr_bh_pct", "spotup_pct", "postup_pct",
    "cut_pct", "transition_pct",
    "fga_pct_restricted", "fga_pct_midrange", "fga_pct_above_break_3",
]

centroids = (
    df_clean
    .group_by("cluster")
    .agg([pl.col(c).mean().alias(c) for c in centroid_cols])
    .sort("cluster")
)

# Print centroids for manual inspection
for row in centroids.iter_rows(named=True):
    c = row["cluster"]
    print(f"\n--- Cluster {c} ---")
    print(f"  Scoring:  pts_p48={row['pts_p48']:.1f}, usg={row['usg_pct']:.1%}, ts={row['ts_pct']:.1%}")
    print(f"  Creation: ast_p48={row['ast_p48']:.1f}, iso={row['iso_pct']:.1%}, pnr_bh={row['pnr_bh_pct']:.1%}")
    print(f"  Interior: reb_p48={row['reb_p48']:.1f}, blk={row['blk_p48']:.1f}, postup={row['postup_pct']:.1%}, restricted={row['fga_pct_restricted']:.1%}")
    print(f"  Perimeter: fg3={row['fg3_pct']:.1%}, spotup={row['spotup_pct']:.1%}, 3pt_fga={row['fga_pct_above_break_3']:.1%}")
    print(f"  Defense:  stl={row['stl_p48']:.1f}, contested={row['contested_shots_p48']:.1f}, defl={row['deflections_p48']:.1f}")
    print(f"  Movement: spd={row['spd']:.2f}, dist={row['dist']:.1f}, tchs={row['tchs_p48']:.1f}")

In [ ]:
# --- Assign archetype names ---
# Note: The exact mapping depends on the cluster centroids above.
# We rank clusters on key dimensions and assign names programmatically.

centroid_np = centroids.select(centroid_cols).to_numpy()

# Score each cluster on defining dimensions
scores = {}
for i, row in enumerate(centroids.iter_rows(named=True)):
    scores[i] = {
        "creation": row["ast_p48"] * 2 + row["pnr_bh_pct"] * 100 + row["iso_pct"] * 100,
        "volume_scoring": row["pts_p48"] * row["usg_pct"] * 100,
        "three_and_d": row["fg3_pct"] * 100 + row["spotup_pct"] * 100 + row["stl_p48"] * 5 - row["usg_pct"] * 50,
        "versatile": row["pts_p48"] + row["reb_p48"] + row["ast_p48"] + row["stl_p48"] + row["blk_p48"],
        "defense_anchor": row["blk_p48"] * 10 + row["contested_shots_p48"] + row["reb_p48"],
        "rim_running": row["fga_pct_restricted"] * 100 + row["cut_pct"] * 100 + row["blk_p48"] * 5,
        "hustle": row["deflections_p48"] + row["screen_assists_p48"] + row["passes_p48"] / 5 - row["pts_p48"] / 10,
        "stretch": row["fga_pct_above_break_3"] * 100 + row["reb_p48"] * 2 - row["spd"] * 10,
    }

ARCHETYPE_NAMES_POOL = [
    ("Primary Creator", "creation"),
    ("Volume Scorer", "volume_scoring"),
    ("3-and-D Wing", "three_and_d"),
    ("Versatile Wing", "versatile"),
    ("Defensive Anchor", "defense_anchor"),
    ("Rim-Running Big", "rim_running"),
    ("Hustle Connector", "hustle"),
    ("Stretch Five", "stretch"),
]

# Greedy assignment: for each archetype, pick the cluster that scores highest
# on that dimension (and hasn't been assigned yet)
assigned = set()
ARCHETYPE_MAP = {}  # cluster_id -> name

# Sort archetype pool by how "distinctive" each scoring dimension is
# (highest max - second highest) to assign most distinctive archetypes first
distinctiveness = []
for name, dim in ARCHETYPE_NAMES_POOL:
    vals = sorted([scores[i][dim] for i in range(K)], reverse=True)
    gap = vals[0] - vals[1] if len(vals) > 1 else vals[0]
    distinctiveness.append((gap, name, dim))
distinctiveness.sort(reverse=True)

for _, name, dim in distinctiveness:
    best_cluster = max(
        (i for i in range(K) if i not in assigned),
        key=lambda i: scores[i][dim]
    )
    ARCHETYPE_MAP[best_cluster] = name
    assigned.add(best_cluster)

# Apply names
archetype_labels = [ARCHETYPE_MAP[c] for c in cluster_labels]
df_clean = df_clean.with_columns(pl.Series("archetype", archetype_labels))

print("Archetype assignments:")
for c in range(K):
    name = ARCHETYPE_MAP[c]
    n = (cluster_labels == c).sum()
    print(f"  Cluster {c} -> {name:20s} ({n:,} player-seasons)")

In [ ]:
# --- Re-plot UMAP with archetype names ---
fig = go.Figure()

for c in range(K):
    name = ARCHETYPE_MAP[c]
    mask = df_clean.filter(pl.col("cluster") == c)
    fig.add_trace(go.Scatter(
        x=mask["umap1"].to_list(),
        y=mask["umap2"].to_list(),
        mode="markers",
        name=name,
        marker=dict(color=ARCHETYPE_COLORS[c], size=5, opacity=0.6),
        text=[
            f"{n} ({s})<br>{name}<br>{p:.1f} pts, {a:.1f} ast, {r:.1f} reb"
            for n, s, p, a, r in zip(
                mask["player_name"].to_list(),
                mask["season_year"].to_list(),
                mask["pts_p48"].to_list(),
                mask["ast_p48"].to_list(),
                mask["reb_p48"].to_list(),
            )
        ],
        hovertemplate="%{text}<extra></extra>",
    ))

fig.update_layout(
    template=TEMPLATE,
    title="The 8 NBA Player Archetypes (UMAP + GMM)",
    xaxis_title="UMAP 1",
    yaxis_title="UMAP 2",
    height=700,
    width=1000,
    legend=dict(x=0.02, y=0.98, font=dict(size=12)),
)
fig.show()

takeaway(
    "The 8 GMM clusters separate cleanly in UMAP space and map to intuitive "
    "basketball archetypes. Traditional positions fail to capture distinctions like "
    "'Stretch Five' vs 'Rim-Running Big' (both listed as C) or 'Primary Creator' vs "
    "'3-and-D Wing' (both listed as G). The tracking and play-type data unique to "
    "this dataset is essential for making these distinctions."
)

### Archetype Justifications

Each archetype name is derived from the cluster centroid statistics:

| Archetype | Defining Features | Why this name? |
|-----------|-------------------|----------------|
| **Primary Creator** | Highest ast/48, PnR ball-handler %, isolation % | These are the lead playmakers who generate offense |
| **Volume Scorer** | Highest pts/48 x usage rate product | High-usage shot creators who score in volume |
| **3-and-D Wing** | High 3PT%, spot-up %, steals; low usage | The modern role player: shoot and defend |
| **Versatile Wing** | Balanced across pts+reb+ast+stl+blk | Do-everything forwards who fill every column |
| **Defensive Anchor** | Highest blocks, contested shots, rebounds | Interior defenders who protect the rim |
| **Rim-Running Big** | Highest restricted-area FGA%, cut %, blocks | Finishers who live at the basket |
| **Hustle Connector** | High deflections, screen assists, passes; low scoring | Glue guys whose value is in effort and connective play |
| **Stretch Five** | High above-break-3 FGA%, rebounds; low speed | Bigs who space the floor with perimeter shooting |

<a id="6"></a>
## 6. Archetype Profiles

Radar charts visualize the signature of each archetype across 8 normalized dimensions.

In [ ]:
# --- Compute radar dimensions per archetype ---
RADAR_DIMS = {
    "Scoring": "pts_p48",
    "Playmaking": "ast_p48",
    "Rebounding": "reb_p48",
    "Defense": "contested_shots_p48",
    "Hustle": "deflections_p48",
    "Speed": "spd",
    "3PT%": "fg3_pct",
    "Usage": "usg_pct",
}

# Compute mean per archetype
radar_data = (
    df_clean
    .group_by("cluster")
    .agg([pl.col(v).mean().alias(k) for k, v in RADAR_DIMS.items()])
    .sort("cluster")
)

# Percentile-rank across clusters (0-100)
categories = list(RADAR_DIMS.keys())
radar_pctile = {}
for dim in categories:
    vals = radar_data[dim].to_numpy()
    # Rank: 0 = lowest, K-1 = highest
    ranks = np.argsort(np.argsort(vals)).astype(float)
    pctile = (ranks / (K - 1)) * 100
    radar_pctile[dim] = pctile

# Build 2x4 radar subplots
fig = make_subplots(
    rows=2, cols=4,
    specs=[[{"type": "polar"}] * 4] * 2,
    subplot_titles=[ARCHETYPE_MAP[c] for c in range(K)],
    vertical_spacing=0.12,
    horizontal_spacing=0.06,
)

for c in range(K):
    row = c // 4 + 1
    col = c % 4 + 1
    values = [radar_pctile[dim][c] for dim in categories]
    values.append(values[0])  # close polygon
    theta = categories + [categories[0]]

    fig.add_trace(
        go.Scatterpolar(
            r=values,
            theta=theta,
            fill="toself",
            fillcolor=ARCHETYPE_COLORS[c],
            opacity=0.4,
            line=dict(color=ARCHETYPE_COLORS[c], width=2),
            name=ARCHETYPE_MAP[c],
            showlegend=False,
        ),
        row=row, col=col,
    )

fig.update_polars(
    radialaxis=dict(range=[0, 100], ticksuffix="%", showticklabels=False),
    angularaxis=dict(tickfont=dict(size=9)),
)

fig.update_layout(
    template=TEMPLATE,
    title="Archetype Radar Profiles (Percentile Rank Across Clusters)",
    height=700,
    width=1100,
)
fig.show()

In [ ]:
# --- Example players per archetype ---

# For each archetype, find the 5 most representative players
# (closest to cluster centroid in scaled feature space)
centroid_means = gmm_final.means_  # shape: (K, n_features)

print("Top 5 Example Players Per Archetype")
print("=" * 60)

for c in range(K):
    name = ARCHETYPE_MAP[c]
    cluster_mask = cluster_labels == c
    cluster_X = X_scaled[cluster_mask]
    cluster_df = df_clean.filter(pl.col("cluster") == c)

    # Distance to centroid
    dists = np.linalg.norm(cluster_X - centroid_means[c], axis=1)
    top5_idx = np.argsort(dists)[:5]

    print(f"\n{name}:")
    for idx in top5_idx:
        row = cluster_df.row(idx, named=True)
        print(
            f"  {row['player_name']:25s} ({row['season_year']}) "
            f"- {row['pts_p48']:.1f}p/{row['reb_p48']:.1f}r/{row['ast_p48']:.1f}a per 48"
        )

takeaway(
    "The radar profiles show distinct signatures: Primary Creators peak in Playmaking "
    "and Usage, Defensive Anchors dominate Defense and Rebounding, 3-and-D Wings spike "
    "in 3PT% with low Usage, and Hustle Connectors excel in Hustle metrics despite low "
    "Scoring. The example players validate these labels — they match the basketball "
    "intuition of scouts and analysts."
)

<a id="7"></a>
## 7. Era Analysis: How Archetypes Have Shifted

The NBA has undergone a tactical revolution. Have the archetypes shifted in frequency
over the tracking era (2013–2025)?

In [ ]:
# --- Count players per archetype per season ---
era_counts = (
    df_clean
    .group_by(["season_year", "archetype"])
    .agg(pl.count().alias("n_players"))
    .sort(["season_year", "archetype"])
)

# Pivot for stacked area
archetype_names = [ARCHETYPE_MAP[c] for c in range(K)]
seasons = sorted(era_counts["season_year"].unique().to_list())

fig = go.Figure()

for c in range(K):
    name = ARCHETYPE_MAP[c]
    subset = era_counts.filter(pl.col("archetype") == name).sort("season_year")

    # Fill missing seasons with 0
    season_counts = {}
    for row in subset.iter_rows(named=True):
        season_counts[row["season_year"]] = row["n_players"]
    counts = [season_counts.get(s, 0) for s in seasons]

    fig.add_trace(go.Scatter(
        x=seasons,
        y=counts,
        name=name,
        mode="lines",
        stackgroup="one",
        line=dict(width=0.5, color=ARCHETYPE_COLORS[c]),
        fillcolor=ARCHETYPE_COLORS[c],
        hovertemplate=f"{name}<br>Season: %{{x}}<br>Players: %{{y}}<extra></extra>",
    ))

fig.update_layout(
    template=TEMPLATE,
    title="The Shifting NBA: Player Archetype Distribution Over Time",
    xaxis_title="Season",
    yaxis_title="Number of Players",
    height=600,
    width=1000,
    legend=dict(x=1.02, y=1, font=dict(size=11)),
    hovermode="x unified",
)
fig.show()

takeaway(
    "The stacked area chart reveals the NBA's tactical evolution in quantitative terms. "
    "Key trends: the rise of 3-and-D Wings and Stretch Fives as the league embraced "
    "perimeter spacing, the decline of traditional post-up Defensive Anchors, and the "
    "steady demand for Primary Creators who can run pick-and-roll. The 'positionless "
    "basketball' narrative is not just a talking point — it shows up clearly in how "
    "the archetype distribution has flattened over time."
)

<a id="8"></a>
## 8. Player Comparisons

Three pairs of players who look similar on the box score but land in different
archetypes — proof that tracking, hustle, and play-type data reveal what box scores hide.

In [ ]:
# --- Build comparison pairs ---
# We'll pick well-known players and find their most recent season in the dataset

COMPARISON_PAIRS = [
    ("LeBron James", "Jimmy Butler",
     "Both average ~25 pts and ~7 reb, but LeBron's creation (PnR, iso) vs "
     "Butler's hustle (deflections, charges drawn) place them in different archetypes."),
    ("Nikola Jokic", "Rudy Gobert",
     "Both are centers who dominate the paint, but Jokic's playmaking (7+ ast) "
     "and perimeter shooting make him a different archetype than Gobert's pure "
     "rim protection."),
    ("Devin Booker", "Mikal Bridges",
     "Both are wings on the same team, but Booker's high-usage scoring isolations "
     "vs Bridges' 3-and-D spot-up role put them in opposite corners of the feature space."),
]


def get_player_latest(name: str) -> dict | None:
    """Get most recent season for a player."""
    matches = df_clean.filter(
        pl.col("player_name").str.contains(name)
    ).sort("season_year", descending=True)
    if matches.height == 0:
        return None
    return matches.row(0, named=True)


# Build radar overlay for each pair
fig = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "polar"}] * 3],
    subplot_titles=[
        f"{p1} vs {p2}" for p1, p2, _ in COMPARISON_PAIRS
    ],
    horizontal_spacing=0.08,
)

pair_colors = [COLORS["blue"], COLORS["red"]]

for col_idx, (name1, name2, narrative) in enumerate(COMPARISON_PAIRS, 1):
    p1 = get_player_latest(name1)
    p2 = get_player_latest(name2)

    if p1 is None or p2 is None:
        print(f"Could not find data for {name1} or {name2}, skipping.")
        continue

    # Compute percentile values for each player among all player-seasons
    for i, (player, name) in enumerate([(p1, name1), (p2, name2)]):
        values = []
        for dim_name, col in RADAR_DIMS.items():
            all_vals = df_clean[col].drop_nulls().to_numpy()
            pctile = (all_vals < player[col]).sum() / len(all_vals) * 100
            values.append(pctile)
        values.append(values[0])  # close
        theta = categories + [categories[0]]

        fig.add_trace(
            go.Scatterpolar(
                r=values,
                theta=theta,
                fill="toself",
                opacity=0.3,
                line=dict(color=pair_colors[i], width=2),
                name=f"{name} ({player['archetype']})",
                legendgroup=f"pair{col_idx}",
                showlegend=True,
            ),
            row=1, col=col_idx,
        )

    print(f"\nPair {col_idx}: {name1} ({p1['archetype']}, {p1['season_year']}) "
          f"vs {name2} ({p2['archetype']}, {p2['season_year']})")
    print(f"  {narrative}")

fig.update_polars(
    radialaxis=dict(range=[0, 100], ticksuffix="%", showticklabels=False),
    angularaxis=dict(tickfont=dict(size=9)),
)

fig.update_layout(
    template=TEMPLATE,
    title="Player Comparisons: Same Box Score, Different Archetype",
    height=500,
    width=1200,
    legend=dict(font=dict(size=10)),
)
fig.show()

In [ ]:
# --- Detailed stat comparison table ---
compare_cols = [
    "player_name", "season_year", "archetype", "pos_group",
    "pts_p48", "reb_p48", "ast_p48", "stl_p48", "blk_p48",
    "fg3_pct", "ts_pct", "usg_pct", "spd",
    "contested_shots_p48", "deflections_p48",
    "iso_pct", "pnr_bh_pct", "spotup_pct",
]

comparison_rows = []
for name1, name2, _ in COMPARISON_PAIRS:
    p1 = get_player_latest(name1)
    p2 = get_player_latest(name2)
    if p1 and p2:
        comparison_rows.append({k: p1.get(k) for k in compare_cols})
        comparison_rows.append({k: p2.get(k) for k in compare_cols})

if comparison_rows:
    comparison_df = pl.DataFrame(comparison_rows)
    display(comparison_df)

takeaway(
    "These comparisons demonstrate why traditional box scores are insufficient. "
    "LeBron and Butler both put up big numbers, but their *how* is completely different: "
    "LeBron creates through PnR and isolation, Butler through hustle and transition. "
    "Jokic and Gobert are both centers, but Jokic's passing makes him a Primary Creator. "
    "Booker and Bridges play together but occupy opposite archetypes. The tracking and "
    "synergy data in this dataset makes these distinctions possible."
)

<a id="9"></a>
## 9. Conclusion & Cross-Links

In [ ]:
# Cleanup handled by nbadb_utils.get_connection() via atexit
print("Analysis complete.")

### Summary

This notebook demonstrated that **tracking data, hustle stats, and play-type profiles
reveal player archetypes that traditional positions cannot capture**:

1. **Feature construction** from 3 data sources yields ~35 features per player-season
2. **PCA** reveals the latent structure: creation vs role, perimeter vs interior, offense vs defense
3. **UMAP** shows that traditional G/F/C positions overlap massively in feature space
4. **GMM clustering** discovers 8 distinct archetypes that map to basketball intuition
5. **Era analysis** quantifies the tactical revolution: rise of 3-and-D wings, decline of post-up bigs
6. **Player comparisons** prove that box scores alone cannot distinguish fundamentally different player types

The key enabler is data unique to the `wyattowalsh/basketball` dataset: speed, distance, touches,
contested shots, deflections, play-type possession shares, and shot zone profiles. Without these
features, we would be limited to the same box-score clustering that has been done a thousand times.

---

### nbadb Notebook Suite

| Part | Notebook | Description |
|---|---|---|
| Part 1 | [MVP Prediction](./nba_mvp_predictor.ipynb) | MVP Prediction with Tracking & Synergy Data |
| **Part 2** | **Player Archetypes** (this notebook) | **Data-Driven Player Archetypes (UMAP + GMM)** |
| Part 3 | [Game Outcome Prediction](./nba_game_prediction.ipynb) | Game Outcome Prediction (Stacking Ensemble) |
| Part 4 | [Draft Combine to Career](./nba_draft_combine_analysis.ipynb) | Draft Combine to Career Prediction |
| Part 5 | [Defense Decoded](./nba_defense_decoded.ipynb) | Defense Decoded (Tracking + Hustle + Synergy PCA) |
| Part 6 | [Interactive Player Explorer](./nba_player_dashboard.ipynb) | Interactive Player Explorer |
| Part 7 | [Spatial Shot Analysis](./nba_shot_chart_analysis.ipynb) | Spatial Shot Analysis & 3-Point Revolution |
| Part 8 | [Player Similarity Engine](./nba_player_similarity.ipynb) | Player Similarity Engine (Cosine + Manhattan) |
| Part 9 | [Career Trajectory](./nba_aging_curves.ipynb) | Career Trajectory & Aging Curve Modeling |
| Part 10 | [Play-by-Play Insights](./nba_play_by_play_insights.ipynb) | Play-by-Play: Win Probability, Runs & Clutch |

---

**Dataset**: [wyattowalsh/basketball on Kaggle](https://www.kaggle.com/datasets/wyattowalsh/basketball)
**Documentation**: [nbadb.dev](https://nbadb.dev)
**Source**: [github.com/wyattowalsh/nbadb](https://github.com/wyattowalsh/nbadb)